# WiDS Wildfire Breakthrough Submission

In [1]:

import os
import gc
import re
import math
import json
import glob
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.special import expit

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

import xgboost as xgb
from xgboost import XGBClassifier

from catboost import CatBoostClassifier
import lightgbm as lgb


warnings.filterwarnings("ignore")

SEED = 20260413
HORIZONS = [12, 24, 48, 72]
INTERVALS = [(0, 12), (12, 24), (24, 48), (48, 72)]


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(SEED)


def find_first(filename: str) -> str:
    search_roots = ["/kaggle/input", "/kaggle/working", ".", "/mnt/data"]
    hits = []
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, files in os.walk(root):
            if filename in files:
                path = os.path.join(dirpath, filename)
                score = len(path)
                low = path.lower()
                if "samplecsv_sub" in low:
                    score += 10_000
                if "old" in low or "backup" in low:
                    score += 500
                hits.append((score, path))
    if not hits:
        raise FileNotFoundError(f"Could not find {filename} under: {search_roots}")
    hits.sort(key=lambda x: (x[0], x[1]))
    return hits[0][1]


def safe_div(a, b, default=0.0):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    out = np.full_like(a, default, dtype=float)
    mask = np.abs(b) > 1e-12
    out[mask] = a[mask] / b[mask]
    out = np.nan_to_num(out, nan=default, posinf=default, neginf=default)
    return out


def monotone_rows(arr):
    arr = np.asarray(arr, dtype=float)
    arr = np.clip(arr, 0.0, 1.0)
    return np.maximum.accumulate(arr, axis=1)


def horizon_label(times, events, horizon):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    y = ((events == 1) & (times <= horizon)).astype(int)
    eligible = ~((events == 0) & (times < horizon))
    return y, eligible


def interval_label(times, events, start, end):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    y = ((events == 1) & (times > start) & (times <= end)).astype(int)
    eligible = (times > start) & ~((events == 0) & (times < end))
    return y, eligible


def harrell_c_index(times, events, risk):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    risk = np.asarray(risk, dtype=float)
    concordant = 0.0
    tied = 0.0
    comparable = 0.0
    n = len(times)
    for i in range(n):
        ti = times[i]
        ei = events[i]
        ri = risk[i]
        for j in range(i + 1, n):
            tj = times[j]
            ej = events[j]
            rj = risk[j]

            if ei == 1 and ti < tj:
                comparable += 1.0
                if ri > rj:
                    concordant += 1.0
                elif ri == rj:
                    tied += 1.0
            elif ej == 1 and tj < ti:
                comparable += 1.0
                if rj > ri:
                    concordant += 1.0
                elif ri == rj:
                    tied += 1.0

    if comparable == 0:
        return 0.5
    return (concordant + 0.5 * tied) / comparable


def brier_at_horizon(times, events, probs, horizon):
    probs = np.asarray(probs, dtype=float)
    y, eligible = horizon_label(times, events, horizon)
    if eligible.sum() == 0:
        return 0.25
    return float(np.mean((probs[eligible] - y[eligible]) ** 2))


def probs_to_expected_time(probs):
    probs = monotone_rows(probs)
    q1 = probs[:, 0]
    q2 = np.clip(probs[:, 1] - probs[:, 0], 0.0, 1.0)
    q3 = np.clip(probs[:, 2] - probs[:, 1], 0.0, 1.0)
    q4 = np.clip(probs[:, 3] - probs[:, 2], 0.0, 1.0)
    q5 = np.clip(1.0 - probs[:, 3], 0.0, 1.0)
    return 6.0 * q1 + 18.0 * q2 + 36.0 * q3 + 60.0 * q4 + 84.0 * q5


def hybrid_score(times, events, probs):
    probs = monotone_rows(probs)
    risk = -probs_to_expected_time(probs)
    c = harrell_c_index(times, events, risk)
    b24 = brier_at_horizon(times, events, probs[:, 1], 24)
    b48 = brier_at_horizon(times, events, probs[:, 2], 48)
    b72 = brier_at_horizon(times, events, probs[:, 3], 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return float(0.3 * c + 0.7 * (1.0 - wb))


def normalize_weights(w):
    w = np.clip(np.asarray(w, dtype=float), 0.0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s


def try_isotonic_fit(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 8 or len(np.unique(y)) < 2 or len(np.unique(x)) < 2:
        const = float(np.mean(y)) if len(y) else 0.5
        return None, const
    ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
    ir.fit(x, y)
    return ir, None


def calibrate_curve(times, events, oof_raw, test_raw):
    oof_raw = monotone_rows(oof_raw)
    test_raw = monotone_rows(test_raw)

    oof_cal = np.zeros_like(oof_raw, dtype=float)
    test_cal = np.zeros_like(test_raw, dtype=float)

    for j, h in enumerate(HORIZONS):
        y, eligible = horizon_label(times, events, h)
        x = oof_raw[eligible, j]
        yy = y[eligible].astype(float)
        model, const = try_isotonic_fit(x, yy)
        if model is None:
            fill = const if const is not None else float(np.mean(yy)) if len(yy) else 0.5
            oof_cal[:, j] = fill
            test_cal[:, j] = fill
        else:
            oof_cal[:, j] = model.predict(oof_raw[:, j])
            test_cal[:, j] = model.predict(test_raw[:, j])

    oof_cal = monotone_rows(oof_cal)
    test_cal = monotone_rows(test_cal)

    raw_score = hybrid_score(times, events, oof_raw)
    cal_score = hybrid_score(times, events, oof_cal)
    if cal_score + 1e-8 >= raw_score:
        return oof_cal, test_cal
    return oof_raw, test_raw


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    def col(name, fill=0.0):
        if name in d.columns:
            return pd.to_numeric(d[name], errors="coerce").astype(float)
        return pd.Series(fill, index=d.index, dtype=float)

    dist = col("dist_min_ci_0_5h").clip(lower=0)
    dist_std = col("dist_std_ci_0_5h").clip(lower=0)
    dist_change = col("dist_change_ci_0_5h")
    dist_slope = col("dist_slope_ci_0_5h")
    closing = col("closing_speed_m_per_h")
    closing_abs = col("closing_speed_abs_m_per_h").abs()
    projected = col("projected_advance_m")
    fit_r2 = col("dist_fit_r2_0_5h").clip(lower=0, upper=1)
    align_abs = col("alignment_abs").clip(lower=0, upper=1)
    align_cos = col("alignment_cos").clip(lower=-1, upper=1)
    cross = col("cross_track_component")
    along = col("along_track_speed")
    area0 = col("area_first_ha").clip(lower=0)
    growth_abs = col("area_growth_abs_0_5h")
    growth_rate = col("area_growth_rate_ha_per_h")
    radial = col("radial_growth_m")
    radial_rate = col("radial_growth_rate_m_per_h")
    centroid_disp = col("centroid_displacement_m")
    centroid_speed = col("centroid_speed_m_per_h")
    num_per = col("num_perimeters_0_5h")
    dt = col("dt_first_last_0_5h")
    low_res = col("low_temporal_resolution_0_5h").clip(lower=0)
    log_area0 = col("log1p_area_first")
    log_growth = col("log1p_growth")
    spread_sin = col("spread_bearing_sin")
    spread_cos = col("spread_bearing_cos")

    close_pos = closing.clip(lower=0)
    along_pos = along.clip(lower=0)
    radial_pos = radial_rate.clip(lower=0)
    projected_pos = projected.clip(lower=0)

    if "event_start_hour" in d.columns:
        hour = col("event_start_hour")
        d["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
        d["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    if "event_start_dayofweek" in d.columns:
        dow = col("event_start_dayofweek")
        d["dow_sin"] = np.sin(2.0 * np.pi * dow / 7.0)
        d["dow_cos"] = np.cos(2.0 * np.pi * dow / 7.0)
    if "event_start_month" in d.columns:
        month = col("event_start_month")
        d["month_sin"] = np.sin(2.0 * np.pi * (month - 1.0) / 12.0)
        d["month_cos"] = np.cos(2.0 * np.pi * (month - 1.0) / 12.0)

    d["dist_km"] = dist / 1000.0
    d["closing_kmh"] = close_pos / 1000.0
    d["projected_advance_km"] = projected_pos / 1000.0
    d["approach_hours"] = np.clip(
        safe_div(dist, close_pos + 0.45 * radial_pos + 0.30 * along_pos + 1.0, default=240.0),
        0.0,
        240.0,
    )
    d["projected_gap_m"] = dist - 0.85 * projected_pos - 0.35 * np.maximum(radial, 0.0) - 0.15 * np.maximum(growth_rate, 0.0)
    d["projected_gap_km"] = d["projected_gap_m"] / 1000.0
    d["dist_over_growth"] = np.clip(
        safe_div(dist, np.abs(radial) + np.abs(projected) + 100.0, default=999.0),
        0.0,
        999.0,
    )
    d["close_align"] = close_pos * align_abs
    d["close_growth"] = close_pos * np.maximum(log_growth, 0.0)
    d["close_growth_rate"] = close_pos * np.maximum(growth_rate, 0.0)
    d["advance_alignment"] = projected_pos * align_abs
    d["spread_toward_evac"] = spread_cos * align_cos
    d["motion_pressure"] = np.log1p(close_pos + 1.0) + 0.75 * np.log1p(along_pos + 1.0) + 0.35 * np.log1p(np.maximum(centroid_speed, 0.0) + 1.0)
    d["growth_pressure"] = np.log1p(area0 + 1.0) + 0.8 * np.log1p(np.maximum(growth_abs, 0.0) + 1.0) + 0.6 * np.log1p(radial_pos + 1.0)
    d["distance_pressure"] = 1.0 / (1.0 + dist / 5000.0)
    d["zone_urgency_proxy"] = d["distance_pressure"] * (1.0 + 0.35 * align_abs + 0.20 * fit_r2) * (1.0 + 0.15 * np.log1p(close_pos + 1.0))
    d["closing_reliability"] = (1.0 - 0.45 * low_res) * (0.5 + 0.5 * fit_r2) * (0.5 + 0.5 * np.minimum(num_per / 3.0, 1.5))
    d["threat_score_eff"] = (
        1.60 * np.log1p(close_pos + 1.0)
        + 0.95 * np.log1p(projected_pos + 1.0)
        + 0.80 * align_abs
        + 0.70 * np.log1p(radial_pos + 1.0)
        + 0.55 * np.log1p(np.maximum(growth_rate, 0.0) + 1.0)
        + 0.35 * fit_r2
        + 0.25 * np.log1p(np.maximum(centroid_speed, 0.0) + 1.0)
        - 1.55 * np.log1p(dist / 1000.0 + 1.0)
        - 0.35 * low_res
    )
    d["threat_score_alt"] = (
        1.20 * d["zone_urgency_proxy"]
        + 0.65 * np.log1p(np.maximum(-dist_change, 0.0) + 1.0)
        + 0.45 * np.log1p(np.maximum(-dist_slope, 0.0) + 1.0)
        + 0.35 * np.log1p(np.maximum(growth_abs, 0.0) + 1.0)
        - 0.45 * np.log1p(d["approach_hours"] + 1.0)
        - 0.20 * np.log1p(dist_std + 1.0)
    )
    d["eta_proxy"] = d["approach_hours"] / (1.0 + 0.35 * align_abs + 0.15 * fit_r2)
    d["near_fire_size"] = np.log1p(area0 + 1.0) / (1.0 + dist / 1000.0)
    d["near_growth_size"] = np.log1p(np.maximum(growth_abs, 0.0) + 1.0) / (1.0 + dist / 1000.0)
    d["uncertainty_penalty"] = low_res + 0.25 * (1.0 - fit_r2) + 0.15 * safe_div(dist_std, dist + 100.0, default=0.0)
    d["perimeter_density"] = safe_div(num_per, dt + 0.25, default=0.0)
    d["cross_vs_along"] = safe_div(np.abs(cross), np.abs(along) + 50.0, default=0.0)
    d["relative_motion_to_gap"] = safe_div(np.abs(centroid_disp), dist + 100.0, default=0.0)
    d["dist_close_interaction"] = d["distance_pressure"] * np.log1p(close_pos + 1.0)
    d["align_growth_interaction"] = align_abs * np.log1p(radial_pos + 1.0)
    d["threat_minus_uncertainty"] = d["threat_score_eff"] - d["uncertainty_penalty"]

    if "hour_cos" in d.columns:
        d["late_shift_risk"] = d["hour_cos"] * d["distance_pressure"]
    if "month_sin" in d.columns:
        d["season_pressure"] = d["month_sin"] * d["growth_pressure"]

    for c in d.columns:
        if c == "event_id":
            continue
        d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d.replace([np.inf, -np.inf], np.nan)
    return d


def get_feature_sets(train_df: pd.DataFrame, test_df: pd.DataFrame):
    common = [c for c in train_df.columns if c in test_df.columns and c != "event_id"]
    numeric = []
    for c in common:
        if pd.api.types.is_numeric_dtype(train_df[c]):
            numeric.append(c)

    preferred_linear = [
        "dist_min_ci_0_5h",
        "dist_km",
        "closing_speed_m_per_h",
        "closing_kmh",
        "projected_advance_m",
        "projected_gap_m",
        "projected_gap_km",
        "approach_hours",
        "alignment_abs",
        "alignment_cos",
        "along_track_speed",
        "cross_track_component",
        "radial_growth_rate_m_per_h",
        "radial_growth_m",
        "area_growth_rate_ha_per_h",
        "area_growth_abs_0_5h",
        "log1p_area_first",
        "log1p_growth",
        "dist_fit_r2_0_5h",
        "low_temporal_resolution_0_5h",
        "centroid_speed_m_per_h",
        "num_perimeters_0_5h",
        "dt_first_last_0_5h",
        "threat_score_eff",
        "threat_score_alt",
        "eta_proxy",
        "zone_urgency_proxy",
        "uncertainty_penalty",
        "distance_pressure",
        "growth_pressure",
        "motion_pressure",
        "hour_sin",
        "hour_cos",
        "month_sin",
        "month_cos",
        "dow_sin",
        "dow_cos",
    ]
    linear = [c for c in preferred_linear if c in numeric]
    if len(linear) < 12:
        linear = numeric[: min(24, len(numeric))]

    knn_pref = [
        "dist_km",
        "closing_kmh",
        "approach_hours",
        "projected_gap_km",
        "alignment_abs",
        "along_track_speed",
        "radial_growth_rate_m_per_h",
        "log1p_area_first",
        "log1p_growth",
        "dist_fit_r2_0_5h",
        "threat_score_eff",
        "threat_score_alt",
        "eta_proxy",
        "zone_urgency_proxy",
        "uncertainty_penalty",
    ]
    knn = [c for c in knn_pref if c in numeric]
    if len(knn) < 8:
        knn = linear[: min(16, len(linear))]

    hazard_pref = [c for c in numeric if any(k in c for k in [
        "dist",
        "close",
        "align",
        "along",
        "projected",
        "threat",
        "growth",
        "radial",
        "zone_urgency",
        "eta",
        "uncertainty",
        "motion",
    ])]
    if len(hazard_pref) < 15:
        hazard_pref = numeric[: min(32, len(numeric))]

    return {
        "all": numeric,
        "linear": linear,
        "knn": knn,
        "hazard": hazard_pref,
        "threat": [c for c in ["threat_score_eff", "threat_score_alt", "eta_proxy", "zone_urgency_proxy", "distance_pressure"] if c in numeric],
    }


def build_stratification_labels(times, events):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)

    labels = np.zeros(len(times), dtype=int)
    labels[(events == 1) & (times > 24)] = 1
    labels[(events == 1) & (times <= 24)] = 2

    counts = pd.Series(labels).value_counts().sort_index()
    min_count = int(counts.min())
    if min_count < 2:
        labels = events.astype(int)
        counts = pd.Series(labels).value_counts().sort_index()
        min_count = int(counts.min())

    n_splits = max(2, min(4, min_count))
    return labels, n_splits


def fit_predict_binary(model_name, X_tr, y_tr, X_va, X_te, seed):
    y_tr = np.asarray(y_tr, dtype=int)
    if len(y_tr) == 0:
        const = 0.5
        return np.full(len(X_va), const), np.full(len(X_te), const)
    uniq = np.unique(y_tr)
    base_rate = float(np.mean(y_tr))
    if len(uniq) < 2:
        return np.full(len(X_va), base_rate), np.full(len(X_te), base_rate)

    pos = int(y_tr.sum())
    neg = int(len(y_tr) - pos)
    pos_w = float(np.clip(neg / max(pos, 1), 1.0, 8.0))
    sample_weight = np.where(y_tr == 1, pos_w, 1.0)

    preds_va = []
    preds_te = []

    if model_name == "cat":
        configs = [
            dict(depth=5, learning_rate=0.035, iterations=700, l2_leaf_reg=6.0, random_strength=0.8, subsample=0.85),
            dict(depth=4, learning_rate=0.025, iterations=900, l2_leaf_reg=8.0, random_strength=1.1, subsample=0.90),
        ]
        for i, cfg in enumerate(configs):
            model = CatBoostClassifier(
                loss_function="Logloss",
                eval_metric="Logloss",
                random_seed=seed + i,
                allow_writing_files=False,
                verbose=False,
                bootstrap_type="Bernoulli",
                thread_count=2,
                **cfg,
            )
            model.fit(X_tr, y_tr, sample_weight=sample_weight)
            preds_va.append(model.predict_proba(X_va)[:, 1])
            preds_te.append(model.predict_proba(X_te)[:, 1])

    elif model_name == "lgb":
        configs = [
            dict(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.85, colsample_bytree=0.85, reg_lambda=1.5, reg_alpha=0.2),
            dict(n_estimators=800, learning_rate=0.022, num_leaves=23, min_child_samples=12, subsample=0.90, colsample_bytree=0.80, reg_lambda=2.0, reg_alpha=0.0),
        ]
        for i, cfg in enumerate(configs):
            model = lgb.LGBMClassifier(
                objective="binary",
                random_state=seed + i,
                n_jobs=2,
                verbosity=-1,
                **cfg,
            )
            model.fit(X_tr, y_tr, sample_weight=sample_weight)
            preds_va.append(model.predict_proba(X_va)[:, 1])
            preds_te.append(model.predict_proba(X_te)[:, 1])

    elif model_name == "xgb":
        configs = [
            dict(n_estimators=500, learning_rate=0.03, max_depth=3, min_child_weight=3, subsample=0.85, colsample_bytree=0.85, reg_lambda=2.0, gamma=0.0),
            dict(n_estimators=700, learning_rate=0.022, max_depth=4, min_child_weight=4, subsample=0.90, colsample_bytree=0.80, reg_lambda=2.8, gamma=0.05),
        ]
        for i, cfg in enumerate(configs):
            model = XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=seed + i,
                tree_method="hist",
                n_jobs=2,
                **cfg,
            )
            model.fit(X_tr, y_tr, sample_weight=sample_weight)
            preds_va.append(model.predict_proba(X_va)[:, 1])
            preds_te.append(model.predict_proba(X_te)[:, 1])

    elif model_name == "logit":
        configs = [
            dict(C=0.60, l1_ratio=0.20),
            dict(C=0.90, l1_ratio=0.35),
        ]
        for i, cfg in enumerate(configs):
            model = Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", RobustScaler()),
                ("clf", LogisticRegression(
                    penalty="elasticnet",
                    solver="saga",
                    max_iter=5000,
                    class_weight=None,
                    random_state=seed + i,
                    **cfg,
                )),
            ])
            model.fit(X_tr, y_tr, clf__sample_weight=sample_weight)
            preds_va.append(model.predict_proba(X_va)[:, 1])
            preds_te.append(model.predict_proba(X_te)[:, 1])

    elif model_name == "knn":
        configs = [7, 11]
        for k in configs:
            model = Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", RobustScaler()),
                ("clf", KNeighborsClassifier(n_neighbors=k, weights="distance", p=2)),
            ])
            model.fit(X_tr, y_tr)
            preds_va.append(model.predict_proba(X_va)[:, 1])
            preds_te.append(model.predict_proba(X_te)[:, 1])

    else:
        raise ValueError(f"Unknown model_name={model_name}")

    pred_va = np.mean(np.column_stack(preds_va), axis=1)
    pred_te = np.mean(np.column_stack(preds_te), axis=1)

    pred_va = np.clip(pred_va, 1e-6, 1.0 - 1e-6)
    pred_te = np.clip(pred_te, 1e-6, 1.0 - 1e-6)
    return pred_va, pred_te


def fit_direct_family(model_name, X_train, X_test, feature_cols, times, events, splits):
    Xtr = X_train[feature_cols]
    Xte = X_test[feature_cols]

    n = len(Xtr)
    m = len(Xte)
    raw_oof = np.zeros((n, len(HORIZONS)), dtype=float)
    raw_cnt = np.zeros((n, len(HORIZONS)), dtype=float)
    raw_test = np.zeros((m, len(HORIZONS)), dtype=float)

    for fold_id, (tr_idx, va_idx) in enumerate(splits):
        seed = SEED + 101 * (fold_id + 1) + abs(hash(model_name)) % 10_000
        for j, h in enumerate(HORIZONS):
            y_all, eligible = horizon_label(times, events, h)
            tr_use = tr_idx[eligible[tr_idx]]

            X_tr = Xtr.iloc[tr_use]
            y_tr = y_all[tr_use]
            X_va = Xtr.iloc[va_idx]
            X_te_fold = Xte

            pred_va, pred_te = fit_predict_binary(model_name, X_tr, y_tr, X_va, X_te_fold, seed + 13 * j)

            raw_oof[va_idx, j] += pred_va
            raw_cnt[va_idx, j] += 1.0
            raw_test[:, j] += pred_te / len(splits)

    raw_oof = raw_oof / np.maximum(raw_cnt, 1.0)
    raw_oof = monotone_rows(raw_oof)
    raw_test = monotone_rows(raw_test)

    cal_oof, cal_test = calibrate_curve(times, events, raw_oof, raw_test)
    return np.nan_to_num(cal_oof, nan=0.5), np.nan_to_num(cal_test, nan=0.5)


def fit_threat_isotonic(train_df, test_df, times, events, splits):
    score_train = (
        1.00 * train_df["threat_score_eff"].fillna(train_df["threat_score_eff"].median()).values
        + 0.45 * train_df["threat_score_alt"].fillna(train_df["threat_score_alt"].median()).values
        - 0.25 * np.log1p(train_df["eta_proxy"].fillna(train_df["eta_proxy"].median()).clip(lower=0).values + 1.0)
        + 0.15 * train_df["zone_urgency_proxy"].fillna(train_df["zone_urgency_proxy"].median()).values
    )
    score_test = (
        1.00 * test_df["threat_score_eff"].fillna(train_df["threat_score_eff"].median()).values
        + 0.45 * test_df["threat_score_alt"].fillna(train_df["threat_score_alt"].median()).values
        - 0.25 * np.log1p(test_df["eta_proxy"].fillna(train_df["eta_proxy"].median()).clip(lower=0).values + 1.0)
        + 0.15 * test_df["zone_urgency_proxy"].fillna(train_df["zone_urgency_proxy"].median()).values
    )

    n = len(train_df)
    m = len(test_df)
    oof = np.zeros((n, len(HORIZONS)), dtype=float)
    cnt = np.zeros((n, len(HORIZONS)), dtype=float)
    pred_test = np.zeros((m, len(HORIZONS)), dtype=float)

    for fold_id, (tr_idx, va_idx) in enumerate(splits):
        for j, h in enumerate(HORIZONS):
            y_all, eligible = horizon_label(times, events, h)
            tr_use = tr_idx[eligible[tr_idx]]
            x_tr = score_train[tr_use]
            y_tr = y_all[tr_use].astype(float)

            model, const = try_isotonic_fit(x_tr, y_tr)
            if model is None:
                fill = const if const is not None else float(np.mean(y_tr)) if len(y_tr) else 0.5
                pred_va = np.full(len(va_idx), fill)
                pred_te = np.full(m, fill)
            else:
                pred_va = model.predict(score_train[va_idx])
                pred_te = model.predict(score_test)

            oof[va_idx, j] += pred_va
            cnt[va_idx, j] += 1.0
            pred_test[:, j] += pred_te / len(splits)

    oof = oof / np.maximum(cnt, 1.0)
    return monotone_rows(oof), monotone_rows(pred_test)


def hazards_to_cumulative(h):
    h = np.clip(np.asarray(h, dtype=float), 1e-6, 1.0 - 1e-6)
    p1 = h[:, 0]
    p2 = 1.0 - (1.0 - p1) * (1.0 - h[:, 1])
    p3 = 1.0 - (1.0 - p2) * (1.0 - h[:, 2])
    p4 = 1.0 - (1.0 - p3) * (1.0 - h[:, 3])
    return monotone_rows(np.column_stack([p1, p2, p3, p4]))


def fit_hazard_family(X_train, X_test, feature_cols, times, events, splits):
    Xtr = X_train[feature_cols]
    Xte = X_test[feature_cols]

    n = len(Xtr)
    m = len(Xte)
    oof_h = np.zeros((n, len(INTERVALS)), dtype=float)
    cnt_h = np.zeros((n, len(INTERVALS)), dtype=float)
    test_h = np.zeros((m, len(INTERVALS)), dtype=float)

    for fold_id, (tr_idx, va_idx) in enumerate(splits):
        seed = SEED + 303 * (fold_id + 1)
        for j, (start, end) in enumerate(INTERVALS):
            y_all, eligible = interval_label(times, events, start, end)
            tr_use = tr_idx[eligible[tr_idx]]

            X_tr = Xtr.iloc[tr_use]
            y_tr = y_all[tr_use]
            X_va = Xtr.iloc[va_idx]
            X_te_fold = Xte

            pred_va, pred_te = fit_predict_binary("cat", X_tr, y_tr, X_va, X_te_fold, seed + 17 * j)

            oof_h[va_idx, j] += pred_va
            cnt_h[va_idx, j] += 1.0
            test_h[:, j] += pred_te / len(splits)

    oof_h = oof_h / np.maximum(cnt_h, 1.0)
    raw_oof = hazards_to_cumulative(oof_h)
    raw_test = hazards_to_cumulative(test_h)
    cal_oof, cal_test = calibrate_curve(times, events, raw_oof, raw_test)
    return cal_oof, cal_test


def aft_cdf(z, distribution):
    z = np.asarray(z, dtype=float)
    if distribution == "normal":
        return norm.cdf(z)
    if distribution == "logistic":
        return expit(z)
    if distribution == "extreme":
        return 1.0 - np.exp(-np.exp(z))
    raise ValueError(f"Unsupported AFT distribution: {distribution}")


def fit_aft_family(X_train, X_test, feature_cols, times, events, splits, distribution="logistic", scale=1.1):
    Xtr = X_train[feature_cols]
    Xte = X_test[feature_cols]
    feature_names = list(feature_cols)

    n = len(Xtr)
    m = len(Xte)

    raw_oof = np.zeros(n, dtype=float)
    raw_cnt = np.zeros(n, dtype=float)
    raw_test = np.zeros(m, dtype=float)

    for fold_id, (tr_idx, va_idx) in enumerate(splits):
        seed = SEED + 505 * (fold_id + 1)

        dtr = xgb.DMatrix(Xtr.iloc[tr_idx], feature_names=feature_names)
        dva = xgb.DMatrix(Xtr.iloc[va_idx], feature_names=feature_names)
        dte = xgb.DMatrix(Xte, feature_names=feature_names)

        lower_tr = np.asarray(times[tr_idx], dtype=float)
        upper_tr = np.where(events[tr_idx] == 1, np.asarray(times[tr_idx], dtype=float), np.inf)
        lower_va = np.asarray(times[va_idx], dtype=float)
        upper_va = np.where(events[va_idx] == 1, np.asarray(times[va_idx], dtype=float), np.inf)

        dtr.set_float_info("label_lower_bound", lower_tr)
        dtr.set_float_info("label_upper_bound", upper_tr)
        dva.set_float_info("label_lower_bound", lower_va)
        dva.set_float_info("label_upper_bound", upper_va)

        params = {
            "objective": "survival:aft",
            "eval_metric": "aft-nloglik",
            "tree_method": "hist",
            "learning_rate": 0.035,
            "max_depth": 3,
            "min_child_weight": 3.0,
            "subsample": 0.88,
            "colsample_bytree": 0.90,
            "lambda": 1.8,
            "alpha": 0.10,
            "aft_loss_distribution": distribution,
            "aft_loss_distribution_scale": scale,
            "seed": seed,
        }

        booster = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=600,
            evals=[(dva, "valid")],
            early_stopping_rounds=50,
            verbose_eval=False,
        )

        pred_va = booster.predict(dva)
        pred_te = booster.predict(dte)

        raw_oof[va_idx] += pred_va
        raw_cnt[va_idx] += 1.0
        raw_test += pred_te / len(splits)

    raw_oof = raw_oof / np.maximum(raw_cnt, 1.0)

    raw_oof_probs = []
    raw_test_probs = []
    for h in HORIZONS:
        z_oof = (np.log(h) - raw_oof) / scale
        z_test = (np.log(h) - raw_test) / scale
        raw_oof_probs.append(aft_cdf(z_oof, distribution))
        raw_test_probs.append(aft_cdf(z_test, distribution))

    raw_curve_oof = monotone_rows(np.column_stack(raw_oof_probs))
    raw_curve_test = monotone_rows(np.column_stack(raw_test_probs))
    param_oof, param_test = calibrate_curve(times, events, raw_curve_oof, raw_curve_test)

    risk_oof = -raw_oof
    risk_test = -raw_test
    iso_oof = np.zeros((n, len(HORIZONS)), dtype=float)
    iso_test = np.zeros((m, len(HORIZONS)), dtype=float)

    for j, h in enumerate(HORIZONS):
        y, eligible = horizon_label(times, events, h)
        model, const = try_isotonic_fit(risk_oof[eligible], y[eligible].astype(float))
        if model is None:
            fill = const if const is not None else float(np.mean(y[eligible])) if eligible.sum() else 0.5
            iso_oof[:, j] = fill
            iso_test[:, j] = fill
        else:
            iso_oof[:, j] = model.predict(risk_oof)
            iso_test[:, j] = model.predict(risk_test)

    iso_oof = monotone_rows(iso_oof)
    iso_test = monotone_rows(iso_test)
    return (param_oof, param_test), (iso_oof, iso_test)


def blend_predictions(pred_stack, weights):
    weights = normalize_weights(weights)
    blended = np.tensordot(weights, pred_stack, axes=(0, 0))
    return monotone_rows(blended)


def optimize_weights(pred_stack, times, events, seed, mask=None, n_iter=6000, local_iter=3000):
    pred_stack = np.asarray(pred_stack, dtype=float)
    if mask is None:
        mask = np.ones(len(times), dtype=bool)
    mask = np.asarray(mask, dtype=bool)

    sub_preds = pred_stack[:, mask, :]
    sub_times = np.asarray(times, dtype=float)[mask]
    sub_events = np.asarray(events, dtype=int)[mask]
    n_models = pred_stack.shape[0]

    rng = np.random.default_rng(seed)

    def score_w(w):
        pred = blend_predictions(sub_preds, w)
        return hybrid_score(sub_times, sub_events, pred)

    best_w = np.ones(n_models, dtype=float) / n_models
    best_score = score_w(best_w)

    expert_scores = []
    for i in range(n_models):
        w = np.zeros(n_models, dtype=float)
        w[i] = 1.0
        sc = score_w(w)
        expert_scores.append(sc)
        if sc > best_score:
            best_score = sc
            best_w = w.copy()

    order = np.argsort(expert_scores)[::-1]
    for k in range(2, min(6, n_models) + 1):
        w = np.zeros(n_models, dtype=float)
        w[order[:k]] = 1.0
        w /= w.sum()
        sc = score_w(w)
        if sc > best_score:
            best_score = sc
            best_w = w.copy()

    alphas = [0.15, 0.30, 0.60, 1.0, 2.5, 5.0]
    loops_each = max(250, n_iter // len(alphas))
    for alpha in alphas:
        for _ in range(loops_each):
            w = rng.dirichlet(np.full(n_models, alpha))
            if n_models > 3 and rng.random() < 0.80:
                sparsity = rng.uniform(0.25, 0.80)
                keep = rng.random(n_models) < sparsity
                if not keep.any():
                    keep[rng.integers(0, n_models)] = True
                w = w * keep
                w = normalize_weights(w)
            sc = score_w(w)
            if sc > best_score:
                best_score = sc
                best_w = w.copy()

    for scale in [0.10, 0.05, 0.02, 0.01]:
        for _ in range(max(250, local_iter // 4)):
            w = normalize_weights(best_w + rng.normal(0.0, scale, size=n_models))
            sc = score_w(w)
            if sc > best_score:
                best_score = sc
                best_w = w.copy()

    return normalize_weights(best_w), float(best_score)


def compute_gate(train_df, test_df):
    cols = ["threat_score_eff", "threat_score_alt", "eta_proxy", "zone_urgency_proxy", "dist_km", "approach_hours"]
    train_vals = []
    test_vals = []

    for c in cols:
        tr = train_df[c].astype(float).values
        te = test_df[c].astype(float).values
        mu = np.nanmean(tr)
        sd = np.nanstd(tr)
        sd = sd if sd > 1e-8 else 1.0
        tr_z = (np.nan_to_num(tr, nan=mu) - mu) / sd
        te_z = (np.nan_to_num(te, nan=mu) - mu) / sd
        train_vals.append(tr_z)
        test_vals.append(te_z)

    tr_z1, tr_z2, tr_z3, tr_z4, tr_z5, tr_z6 = train_vals
    te_z1, te_z2, te_z3, te_z4, te_z5, te_z6 = test_vals

    gate_train_score = 0.45 * tr_z1 + 0.25 * tr_z2 + 0.20 * tr_z4 - 0.20 * tr_z3 - 0.18 * tr_z5 - 0.12 * tr_z6
    gate_test_score = 0.45 * te_z1 + 0.25 * te_z2 + 0.20 * te_z4 - 0.20 * te_z3 - 0.18 * te_z5 - 0.12 * te_z6

    gate_train = expit(1.30 * gate_train_score)
    gate_test = expit(1.30 * gate_test_score)
    return gate_train, gate_test


def validate_submission(sub, sample_ids):
    expected_cols = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    assert list(sub.columns) == expected_cols, f"Columns mismatch: {sub.columns.tolist()}"
    assert sub["event_id"].is_unique, "Duplicate event_id detected"
    assert sub["event_id"].tolist() == list(sample_ids), "event_id order/content does not match sample submission"
    prob_cols = expected_cols[1:]
    assert np.isfinite(sub[prob_cols].to_numpy()).all(), "Non-finite probabilities detected"
    assert ((sub[prob_cols] >= 0.0) & (sub[prob_cols] <= 1.0)).all().all(), "Probabilities must be in [0, 1]"
    assert (sub["prob_12h"] <= sub["prob_24h"]).all(), "Monotonicity violation 12h>24h"
    assert (sub["prob_24h"] <= sub["prob_48h"]).all(), "Monotonicity violation 24h>48h"
    assert (sub["prob_48h"] <= sub["prob_72h"]).all(), "Monotonicity violation 48h>72h"


train_path = find_first("train.csv")
test_path = find_first("test.csv")
sample_path = find_first("sample_submission.csv")

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

feature_cols_base = [c for c in train_raw.columns if c not in ["event_id", "time_to_hit_hours", "event"] and c in test_raw.columns]

train_feat = engineer_features(train_raw[["event_id"] + feature_cols_base].copy())
test_feat = engineer_features(test_raw[["event_id"] + feature_cols_base].copy())

feature_sets = get_feature_sets(train_feat, test_feat)

times = train_raw["time_to_hit_hours"].astype(float).values
events = train_raw["event"].astype(int).values

strat_labels, n_splits = build_stratification_labels(times, events)
n_repeats = 6 if n_splits >= 3 else 8
splits_full = list(RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=SEED).split(np.zeros(len(train_raw)), strat_labels))
splits_medium = splits_full[: max(n_splits * 4, len(splits_full) // 2)]
splits_light = splits_full[: max(n_splits * 3, len(splits_full) // 2)]

expert_oof = {}
expert_test = {}

oof_cat, test_cat = fit_direct_family("cat", train_feat, test_feat, feature_sets["all"], times, events, splits_full)
expert_oof["cat_direct"] = oof_cat
expert_test["cat_direct"] = test_cat

oof_lgb, test_lgb = fit_direct_family("lgb", train_feat, test_feat, feature_sets["all"], times, events, splits_full)
expert_oof["lgb_direct"] = oof_lgb
expert_test["lgb_direct"] = test_lgb

oof_xgb, test_xgb = fit_direct_family("xgb", train_feat, test_feat, feature_sets["all"], times, events, splits_medium)
expert_oof["xgb_direct"] = oof_xgb
expert_test["xgb_direct"] = test_xgb

oof_logit, test_logit = fit_direct_family("logit", train_feat, test_feat, feature_sets["linear"], times, events, splits_full)
expert_oof["logit_direct"] = oof_logit
expert_test["logit_direct"] = test_logit

oof_knn, test_knn = fit_direct_family("knn", train_feat, test_feat, feature_sets["knn"], times, events, splits_full)
expert_oof["knn_direct"] = oof_knn
expert_test["knn_direct"] = test_knn

oof_threat, test_threat = fit_threat_isotonic(train_feat, test_feat, times, events, splits_full)
expert_oof["threat_isotonic"] = oof_threat
expert_test["threat_isotonic"] = test_threat

oof_hazard, test_hazard = fit_hazard_family(train_feat, test_feat, feature_sets["hazard"], times, events, splits_medium)
expert_oof["hazard_cat"] = oof_hazard
expert_test["hazard_cat"] = test_hazard

(aft_log_param_oof, aft_log_param_test), (aft_log_iso_oof, aft_log_iso_test) = fit_aft_family(
    train_feat, test_feat, feature_sets["all"], times, events, splits_light, distribution="logistic", scale=1.10
)
expert_oof["aft_logistic_param"] = aft_log_param_oof
expert_test["aft_logistic_param"] = aft_log_param_test
expert_oof["aft_logistic_iso"] = aft_log_iso_oof
expert_test["aft_logistic_iso"] = aft_log_iso_test

(aft_norm_param_oof, aft_norm_param_test), (_, _) = fit_aft_family(
    train_feat, test_feat, feature_sets["all"], times, events, splits_light, distribution="normal", scale=1.00
)
expert_oof["aft_normal_param"] = aft_norm_param_oof
expert_test["aft_normal_param"] = aft_norm_param_test

expert_names = list(expert_oof.keys())
pred_stack_train = np.stack([expert_oof[name] for name in expert_names], axis=0)
pred_stack_test = np.stack([expert_test[name] for name in expert_names], axis=0)

expert_scores = {name: hybrid_score(times, events, expert_oof[name]) for name in expert_names}
expert_scores = dict(sorted(expert_scores.items(), key=lambda kv: kv[1], reverse=True))

global_w, global_score = optimize_weights(pred_stack_train, times, events, seed=SEED + 9001)
global_oof = blend_predictions(pred_stack_train, global_w)
global_test = blend_predictions(pred_stack_test, global_w)

gate_train, gate_test = compute_gate(train_feat, test_feat)
near_mask = gate_train >= np.quantile(gate_train, 0.55)
far_mask = ~near_mask

final_oof = global_oof.copy()
final_test = global_test.copy()
final_name = "global"

if near_mask.sum() >= 50 and far_mask.sum() >= 50 and events[near_mask].sum() >= 8 and events[far_mask].sum() >= 8:
    near_w_opt, _ = optimize_weights(pred_stack_train, times, events, seed=SEED + 9101, mask=near_mask, n_iter=4000, local_iter=2000)
    far_w_opt, _ = optimize_weights(pred_stack_train, times, events, seed=SEED + 9201, mask=far_mask, n_iter=4000, local_iter=2000)

    near_w = normalize_weights(0.65 * global_w + 0.35 * near_w_opt)
    far_w = normalize_weights(0.65 * global_w + 0.35 * far_w_opt)

    near_oof = blend_predictions(pred_stack_train, near_w)
    far_oof = blend_predictions(pred_stack_train, far_w)
    gated_oof = monotone_rows(gate_train[:, None] * near_oof + (1.0 - gate_train)[:, None] * far_oof)

    near_test = blend_predictions(pred_stack_test, near_w)
    far_test = blend_predictions(pred_stack_test, far_w)
    gated_test = monotone_rows(gate_test[:, None] * near_test + (1.0 - gate_test)[:, None] * far_test)

    if hybrid_score(times, events, gated_oof) > global_score + 1e-5:
        final_oof = gated_oof
        final_test = gated_test
        final_name = "gated"

final_oof, final_test = calibrate_curve(times, events, final_oof, final_test)
final_oof = monotone_rows(final_oof)
final_test = monotone_rows(final_test)

final_score = hybrid_score(times, events, final_oof)

print("Files used:")
print(" train:", train_path)
print(" test :", test_path)
print(" sample:", sample_path)
print()
print("Expert OOF hybrid scores:")
for name, sc in expert_scores.items():
    print(f" {name:20s} {sc:.6f}")
print()
print(f"Global blend OOF hybrid: {global_score:.6f}")
print(f"Chosen blend mode     : {final_name}")
print(f"Final blend OOF hybrid: {final_score:.6f}")
print("Blend weights:")
for name, w in sorted(zip(expert_names, global_w), key=lambda x: x[1], reverse=True):
    print(f" {name:20s} {w:.5f}")

submission = pd.DataFrame({
    "event_id": test_raw["event_id"].values,
    "prob_12h": final_test[:, 0],
    "prob_24h": final_test[:, 1],
    "prob_48h": final_test[:, 2],
    "prob_72h": final_test[:, 3],
})

submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left")
submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] = monotone_rows(
    submission[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].to_numpy()
)

validate_submission(submission, sample_sub["event_id"].tolist())

submission.to_csv("submission.csv", index=False)
with open("blend_diagnostics.json", "w") as f:
    json.dump({
        "expert_scores": expert_scores,
        "global_blend_score": global_score,
        "final_blend_score": final_score,
        "final_mode": final_name,
        "global_weights": {k: float(v) for k, v in zip(expert_names, global_w)},
    }, f, indent=2)

print()
print("submission.csv saved")
print(submission.head())


Files used:
 train: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/train.csv
 test : /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/test.csv
 sample: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26/sample_submission.csv

Expert OOF hybrid scores:
 lgb_direct           0.973762
 cat_direct           0.971403
 aft_logistic_iso     0.971318
 xgb_direct           0.970578
 aft_logistic_param   0.969505
 hazard_cat           0.969147
 threat_isotonic      0.955815
 aft_normal_param     0.946648
 knn_direct           0.944877
 logit_direct         0.913349

Global blend OOF hybrid: 0.974497
Chosen blend mode     : global
Final blend OOF hybrid: 0.976225
Blend weights:
 lgb_direct           0.82006
 cat_direct           0.11585
 logit_direct         0.06409
 xgb_direct           0.00000
 knn_direct           0.00000
 threat_isotonic      0.00000
 hazard_cat           0.00000
 aft_logistic_param   0.00000
 aft_logistic_iso     0.00000
 aft_normal_param     0.00000

su